## Action Responder Prompt

This notebook hopes to:

- Gather trace data for action-responder agent
- Structure and save as mlflow dataset
- Utilize judges to evaluate current agent
- Try with better model to confirm judges are working
- Run GEPA optimzation to improve

In [20]:
import mlflow
import pandas as pd
from mlflow.tracking import MlflowClient

mlflow.tracing.enable_notebook_display()

client = MlflowClient()
experiment = mlflow.get_experiment_by_name("summit-sim")


# Get all traces from last 7 days (adjust as needed)
traces = client.search_traces(
    locations=[experiment.experiment_id],
    filter_string="tags.graph_type = 'simulation' AND tags.agent_name = 'action-responder'",
    max_results=500,
)

In [35]:
spans = [
    span
    for trace in traces
    for span in client.get_trace(trace.info.trace_id).data.spans
    if span.name == "process_player_action"
]

print(f"Found {len(spans)} process_player_action spans")

Found 4 process_player_action spans


[Trace(trace_id=tr-d4b2d4337a4267bd7deab60e93232eac), Trace(trace_id=tr-3bd3e7244e4941acbef2a869b6ca06a8), Trace(trace_id=tr-406318dc83e7db8c112280d8159e8be8), Trace(trace_id=tr-9d781a56f8679d936935fc6155a3be00)]

In [36]:
records = []

for span in spans:
    # Inputs/outputs are stored as JSON strings or dicts
    inputs = span.inputs  # Dict containing student_action, context, etc.
    outputs = span.outputs  # Dict containing DynamicTurnResult fields

    records.append(
        {
            "trace_id": span.trace_id,
            "inputs": inputs,
            "outputs": outputs,
        }
    )

eval_df = pd.DataFrame(records)
eval_df

,trace_id,inputs,outputs
0,tr-d4b2d4337a4267bd7deab60e93232eac,{'state': {'scenario': {'title': 'Falcon Peak ...,"{'action_result': {'was_correct': True, 'compl..."
1,tr-3bd3e7244e4941acbef2a869b6ca06a8,{'state': {'scenario': {'title': 'Falcon Peak ...,"{'action_result': {'was_correct': True, 'compl..."
2,tr-406318dc83e7db8c112280d8159e8be8,{'state': {'scenario': {'title': 'Falcon Peak ...,"{'action_result': {'was_correct': True, 'compl..."
3,tr-9d781a56f8679d936935fc6155a3be00,{'state': {'scenario': {'title': 'Falcon Peak ...,"{'action_result': {'was_correct': True, 'compl..."
